<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/CDL_CBP_TOPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import math
import torch
import torch.nn as nn
import torch.nn.init as init

# ==============================================================================
# Dynamic Derivation of Primes and Lambda
# ==============================================================================
def derive_primes_and_lambda(max_prime=13):
    is_prime = [True] * (max_prime + 1)
    is_prime[0] = is_prime[1] = False
    for p in range(2, int(math.isqrt(max_prime)) + 1):
        if is_prime[p]:
            for i in range(p * p, max_prime + 1, p):
                is_prime[i] = False
    primes = [p for p, valid in enumerate(is_prime) if valid]

    product = 1.0
    for p in primes:
        product *= (1.0 - (1.0 / math.sqrt(p)))

    return primes, 1.0 - product

PRIME_ANCHORS, LAMBDA_DERIVED = derive_primes_and_lambda(max_prime=13)
print(f"Dynamically Derived Primes : {PRIME_ANCHORS}")
print(f"Dynamically Derived Lambda : {LAMBDA_DERIVED:.10f}")


# ==============================================================================
# Topological Governor with Weight-Decay Invariance Lock
# ==============================================================================
class TopologicalGovernor:
    def __init__(self, prime_indices, lam):
        self.prime_indices = prime_indices
        self.lam = lam

    def pre_step(self, model: nn.Module):
        """Attenuates plastic gradient subspace and clamps prime anchor gradients."""
        with torch.no_grad():
            for param in model.parameters():
                if param.grad is None:
                    continue

                param.grad.mul_(self.lam)

                dim = param.dim()
                if dim >= 2:
                    p_valid = [p for p in self.prime_indices if p < param.grad.shape[0]]
                    if p_valid:
                        param.grad[p_valid, :] = 0.0
                elif dim == 1:
                    p_valid = [p for p in self.prime_indices if p < param.grad.shape[0]]
                    if p_valid:
                        param.grad[p_valid] = 0.0

    def post_step(self, model: nn.Module, initial_anchors: dict):
        """Locks prime coordinate values against optimizer weight decay drift."""
        with torch.no_grad():
            model.layer1.linear.weight[self.prime_indices, :] = initial_anchors['l1']
            model.layer2.linear.weight[self.prime_indices, :] = initial_anchors['l2']


# ==============================================================================
# Continual Linear Layer (CBP with Prime Shielding)
# ==============================================================================
class ContinualLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, replacement_rate: float = 1e-4,
                 maturity: int = 100, eta: float = 0.99):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.replacement_rate = replacement_rate
        self.maturity = maturity
        self.eta = eta

        self.linear = nn.Linear(in_features, out_features)
        init.kaiming_uniform_(self.linear.weight, nonlinearity='relu')
        init.zeros_(self.linear.bias)

        self.register_buffer('utility', torch.zeros(out_features))
        self.register_buffer('age', torch.zeros(out_features, dtype=torch.long))
        self.register_buffer('accumulated_replacements', torch.tensor(0.0))
        self.last_activation = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.linear(x)
        self.last_activation = torch.relu(h)
        return self.last_activation

    def update_utility(self, next_layer_weight: torch.Tensor):
        with torch.no_grad():
            if self.last_activation is None:
                return
            mean_act = torch.mean(torch.abs(self.last_activation), dim=0)
            outgoing_weight_sum = torch.sum(torch.abs(next_layer_weight), dim=0)
            instantaneous_utility = mean_act * outgoing_weight_sum
            self.utility.mul_(self.eta).add_(instantaneous_utility, alpha=(1.0 - self.eta))
            self.age.add_(1)

    def reinitialize_units(self, next_layer_linear: nn.Linear, protected_indices=None):
        with torch.no_grad():
            eligible_mask = self.age >= self.maturity
            if protected_indices:
                for idx in protected_indices:
                    if idx < self.out_features:
                        eligible_mask[idx] = False

            n_eligible = eligible_mask.sum().item()
            if n_eligible == 0:
                return

            self.accumulated_replacements.add_(n_eligible * self.replacement_rate)

            while self.accumulated_replacements >= 1.0:
                masked_utility = torch.where(
                    eligible_mask,
                    self.utility,
                    torch.tensor(float('inf'), device=self.utility.device)
                )
                r = torch.argmin(masked_utility).item()

                bound = (1.0 / self.in_features) ** 0.5
                init.uniform_(self.linear.weight[r, :], -bound, bound)
                self.linear.bias[r].zero_()

                next_layer_linear.weight[:, r].zero_()

                self.utility[r] = 0.0
                self.age[r] = 0
                eligible_mask[r] = False
                self.accumulated_replacements.sub_(1.0)


# ==============================================================================
# Governed Network
# ==============================================================================
class GovernedContinualMLP(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int,
                 replacement_rate: float = 1e-3, maturity: int = 50):
        super().__init__()
        self.layer1 = ContinualLinear(in_dim, hidden_dim, replacement_rate, maturity)
        self.layer2 = ContinualLinear(hidden_dim, hidden_dim, replacement_rate, maturity)
        self.head = nn.Linear(hidden_dim, out_dim)
        init.kaiming_uniform_(self.head.weight, nonlinearity='linear')
        init.zeros_(self.head.bias)

        self.governor = TopologicalGovernor(PRIME_ANCHORS, LAMBDA_DERIVED)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h1 = self.layer1(x)
        h2 = self.layer2(h1)
        return self.head(h2)

    def govern_pre_step(self):
        self.governor.pre_step(self)

    def govern_post_step(self, initial_anchors: dict):
        self.governor.post_step(self, initial_anchors)

    def cbp_step(self):
        self.layer1.update_utility(self.layer2.linear.weight)
        self.layer2.update_utility(self.head.weight)
        self.layer1.reinitialize_units(self.layer2.linear, protected_indices=PRIME_ANCHORS)
        self.layer2.reinitialize_units(self.head, protected_indices=PRIME_ANCHORS)

    def verify_prime_invariance(self, initial_anchors: dict) -> float:
        with torch.no_grad():
            w1_drift = torch.max(torch.abs(self.layer1.linear.weight[PRIME_ANCHORS, :] - initial_anchors['l1'])).item()
            w2_drift = torch.max(torch.abs(self.layer2.linear.weight[PRIME_ANCHORS, :] - initial_anchors['l2'])).item()
            return max(w1_drift, w2_drift)


# ==============================================================================
# Online Verification Run
# ==============================================================================
if __name__ == "__main__":
    torch.manual_seed(42)

    in_dim, hidden_dim, out_dim = 64, 128, 10
    model = GovernedContinualMLP(in_dim, hidden_dim, out_dim, replacement_rate=1e-3, maturity=30)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    initial_anchors = {
        'l1': model.layer1.linear.weight[PRIME_ANCHORS, :].clone(),
        'l2': model.layer2.linear.weight[PRIME_ANCHORS, :].clone()
    }

    print("\n--- Executing Invariance Verification ---")
    current_center = torch.zeros(in_dim)

    for step in range(1, 1001):
        if step % 250 == 1:
            current_center = torch.randn(in_dim)
            task_id = (step // 250) + 1
            print(f"\n[Environment Shift] Task {task_id} Active at Step {step}")

        x = torch.randn(32, in_dim) + current_center
        y = torch.randint(0, out_dim, (32,))

        optimizer.zero_grad()
        outputs = model(x)
        loss = criterion(outputs, y)
        loss.backward()

        # Enforce gradient attenuation & clamp prime gradients
        model.govern_pre_step()
        optimizer.step()
        # Enforce exact coordinate retention against weight decay
        model.govern_post_step(initial_anchors)
        # Continual Backprop step with shielded primes
        model.cbp_step()

        if step % 250 == 0:
            drift = model.verify_prime_invariance(initial_anchors)
            l1_u = model.layer1.utility.mean().item()
            l2_u = model.layer2.utility.mean().item()
            print(f"Step {step:4d} | Loss: {loss.item():.4f} | Prime Anchor Drift: {drift:.10f} | Utility (L1/L2): {l1_u:.4f} / {l2_u:.4f}")

Dynamically Derived Primes : [2, 3, 5, 7, 11, 13]
Dynamically Derived Lambda : 0.9785142874

--- Executing Invariance Verification ---

[Environment Shift] Task 1 Active at Step 1
Step  250 | Loss: 2.3079 | Prime Anchor Drift: 0.0000000000 | Utility (L1/L2): 9.0760 / 0.4613

[Environment Shift] Task 2 Active at Step 251
Step  500 | Loss: 2.3874 | Prime Anchor Drift: 0.0000000000 | Utility (L1/L2): 10.2141 / 0.4392

[Environment Shift] Task 3 Active at Step 501
Step  750 | Loss: 2.4041 | Prime Anchor Drift: 0.0000000000 | Utility (L1/L2): 10.2805 / 0.3516

[Environment Shift] Task 4 Active at Step 751
Step 1000 | Loss: 2.4367 | Prime Anchor Drift: 0.0000000000 | Utility (L1/L2): 10.4019 / 0.3778


In [1]:
!nvidia-smi

Fri Sep 25 21:55:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   63C    P8             17W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import math
import random
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from datasets import load_dataset

# ==============================================================================
# 0. Deterministic Reproducibility Configuration (Seed 123)
# ==============================================================================

SEED = 123
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f"Deterministic execution seed locked to: {SEED}")


# ==============================================================================
# 1. Dynamic Number-Theoretic Invariant Generators (No Hardcoding)
# Lambda = 1 - \prod_{p \in P} (1 - 1 / \sqrt{p})
# ==============================================================================

def derive_primes_and_lambda(max_prime=13):
    is_prime = [True] * (max_prime + 1)
    is_prime[0] = is_prime[1] = False
    for p in range(2, int(math.isqrt(max_prime)) + 1):
        if is_prime[p]:
            for i in range(p * p, max_prime + 1, p):
                is_prime[i] = False
    primes = [p for p, valid in enumerate(is_prime) if valid]

    product = 1.0
    for p in primes:
        product *= (1.0 - (1.0 / math.sqrt(p)))

    return primes, 1.0 - product

PRIME_ANCHORS, LAMBDA_DERIVED = derive_primes_and_lambda(max_prime=13)
print(f"Dynamically Derived Primes : {PRIME_ANCHORS}")
print(f"Dynamically Derived Lambda : {LAMBDA_DERIVED:.10f}")


# ==============================================================================
# 2. Production LLM Topological Governor Hook
# ==============================================================================

class LLMTopologicalGovernor:
    """
    Enforces prime-coordinate invariance across vocabulary embedding matrices
    and classifier heads while attenuating the complementary plastic subspace.
    """
    def __init__(self, prime_indices, lam):
        self.prime_indices = prime_indices
        self.lam = lam

    def pre_step(self, model: torch.nn.Module):
        """Attenuates plastic gradient subspace and clamps prime anchor gradients."""
        with torch.no_grad():
            for name, param in model.named_parameters():
                if param.grad is None:
                    continue

                # Attenuate plastic gradient space
                param.grad.mul_(self.lam)

                # Zero gradients at prime coordinates on embedding and head layers
                if any(k in name.lower() for k in ["embed", "wte", "lm_head"]):
                    p_valid = [p for p in self.prime_indices if p < param.grad.shape[0]]
                    if p_valid:
                        param.grad[p_valid, :] = 0.0

    def post_step(self, model: torch.nn.Module, initial_anchors: dict):
        """Locks prime coordinate values strictly against optimizer weight decay drift."""
        with torch.no_grad():
            for name, param in model.named_parameters():
                if name in initial_anchors:
                    param.data[self.prime_indices, :] = initial_anchors[name]


# ==============================================================================
# 3. Execution Pipeline: Governed Continual Streaming (5,000 Steps)
# ==============================================================================

def run_governed_llm_stream():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_id = "Qwen/Qwen2.5-0.5B"

    print(f"\nInitializing Tokenizer & Model: {model_id}")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
    ).to(device)
    model.train()

    # Locate vocabulary embedding tensor
    target_embed_name = None
    for name, param in model.named_parameters():
        if any(k in name.lower() for k in ["embed_tokens.weight", "wte.weight"]):
            target_embed_name = name
            break

    if target_embed_name is None:
        raise ValueError("Could not find embedding parameter in model hierarchy.")

    print(f"Anchoring Invariant Subspace on Target Tensor: {target_embed_name}")

    # Snapshot baseline prime anchor coordinates directly on device
    initial_anchors = {}
    for name, param in model.named_parameters():
        if name == target_embed_name:
            initial_anchors[name] = param.data[PRIME_ANCHORS, :].clone()

    governor = LLMTopologicalGovernor(PRIME_ANCHORS, LAMBDA_DERIVED)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=1e-4)

    # Stream continuous data from FineWeb-Edu sample
    print("Connecting to live text stream: HuggingFaceFW/fineweb-edu (sample-10BT)...")
    dataset = load_dataset(
        "HuggingFaceFW/fineweb-edu",
        name="sample-10BT",
        split="train",
        streaming=True
    )

    def text_stream():
        for sample in dataset:
            txt = sample.get("text", "").strip()
            if len(txt) > 80:
                yield txt

    stream_iter = iter(text_stream())

    total_steps = 5000
    log_interval = 250

    print(f"\n--- Commencing Governed Continual Streaming on LLM ({total_steps:,} Steps | Seed {SEED}) ---")

    for step in range(1, total_steps + 1):
        batch = [next(stream_iter) for _ in range(4)]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128
        ).to(device)

        inputs["labels"] = inputs["input_ids"].clone()

        optimizer.zero_grad()
        outputs = model(**inputs)
        loss = outputs.loss
        loss.backward()

        # Step 1: Pre-step governor enforcement (zero prime grads, attenuate plastic subspace)
        governor.pre_step(model)

        # Step 2: Plastic parameter updates via optimizer
        optimizer.step()

        # Step 3: Exact coordinate retention against optimizer decay
        governor.post_step(model, initial_anchors)

        if step % log_interval == 0 or step == 1:
            with torch.no_grad():
                current_param = dict(model.named_parameters())[target_embed_name]
                drift = torch.max(
                    torch.abs(current_param[PRIME_ANCHORS, :] - initial_anchors[target_embed_name])
                ).item()
            print(f"Step {step:5d} / {total_steps} | LM Loss: {loss.item():.4f} | Prime Anchor Drift: {drift:.10f}")

    print(f"\nVerification Complete: LLM continual streaming over {total_steps:,} steps operating with exact 0.0000000000 prime anchor drift under seed {SEED}.")


if __name__ == "__main__":
    run_governed_llm_stream()

Deterministic execution seed locked to: 123
Dynamically Derived Primes : [2, 3, 5, 7, 11, 13]
Dynamically Derived Lambda : 0.9785142874

Initializing Tokenizer & Model: Qwen/Qwen2.5-0.5B


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Anchoring Invariant Subspace on Target Tensor: model.embed_tokens.weight
Connecting to live text stream: HuggingFaceFW/fineweb-edu (sample-10BT)...


Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]


--- Commencing Governed Continual Streaming on LLM (5,000 Steps | Seed 123) ---
Step     1 / 5000 | LM Loss: 3.0187 | Prime Anchor Drift: 0.0000000000
Step   250 / 5000 | LM Loss: 3.1059 | Prime Anchor Drift: 0.0000000000
Step   500 / 5000 | LM Loss: 2.6100 | Prime Anchor Drift: 0.0000000000
Step   750 / 5000 | LM Loss: 2.7106 | Prime Anchor Drift: 0.0000000000
Step  1000 / 5000 | LM Loss: 2.6727 | Prime Anchor Drift: 0.0000000000
Step  1250 / 5000 | LM Loss: 2.4076 | Prime Anchor Drift: 0.0000000000
Step  1500 / 5000 | LM Loss: 2.7919 | Prime Anchor Drift: 0.0000000000
Step  1750 / 5000 | LM Loss: 2.9117 | Prime Anchor Drift: 0.0000000000
Step  2000 / 5000 | LM Loss: 2.7832 | Prime Anchor Drift: 0.0000000000
Step  2250 / 5000 | LM Loss: 3.0236 | Prime Anchor Drift: 0.0000000000
Step  2500 / 5000 | LM Loss: 2.8444 | Prime Anchor Drift: 0.0000000000
Step  2750 / 5000 | LM Loss: 3.0683 | Prime Anchor Drift: 0.0000000000
Step  3000 / 5000 | LM Loss: 2.9349 | Prime Anchor Drift: 0.0000000

In [3]:
!nvidia-smi

Fri Sep 25 22:17:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   67C    P0             30W /   72W |    6460MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----